In [1]:
from src.connection.clinets import weaviate_client
from typing import List, Any
import weaviate
import json
import pandas as pd
from langchain_weaviate import WeaviateVectorStore
from sentence_transformers import SentenceTransformer

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class SentenceTransformersEmbeddings:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # returns a list of embeddings for documents
        return self.model.encode(texts).tolist()

    def embed_query(self, text: str) -> List[float]:
        # returns a single embedding for a query
        return self.model.encode([text])[0].tolist()

In [3]:
embedding_model = SentenceTransformersEmbeddings('sentence-transformers/all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 599.04it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [1]:
import pandas as pd

In [2]:
laws = pd.read_csv('dataset/act_raw_text_with_meta.csv')

In [5]:
def get__embeddings(enhanced_query):
    query_embedding = embedding_model.embed_query(enhanced_query)

    return query_embedding

In [16]:
query_embedding = get__embeddings("Pooping in Public Places")

In [7]:
weaviate_client.is_connected()

True

In [17]:
eu = weaviate_client.collections.use("Euro_Laws")
response = eu.query.near_vector(
    near_vector= query_embedding, 
    limit=5
)

for obj in response.objects:
    print(json.dumps(obj.properties, indent=10))  # Inspect the results

{
          "subject_matter": "organisation of transport;  social affairs;  technology and technical regulations;  European construction;  transport policy;  rights and freedoms;  land transport",
          "legal_basis": "31996L0048; 32001L0016",
          "act_name": "2008/164/EC: Commission Decision of 21 December 2007 concerning the technical specification of interoperability relating to persons with reduced mobility in the trans-European conventional and high-speed rail system (notified under document C(2007) 6633) (Text with EEA relevance)",
          "chunk_number": 44,
          "treaty": "TEC (1992)",
          "celex": "32008D0164",
          "document_length": 362537,
          "text": "equipment inside the toilet compartment (except for baby change facilities) shall be operable by exerting a force not exceeding 20 Newtons. There shall be sufficient space inside the toilet compartment to enable a wheelchair as defined in Annex M to be manoeuvred to a position adjacent to the

In [21]:
vectorstore = WeaviateVectorStore(
    client = weaviate_client,
    index_name = "Euro_Laws",
    text_key="text",
    embedding = embedding_model
)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("Drug dealing Sentences")
docs

In [16]:
def search_docs(query):
    celex_ids : list[str,str]
    full_doc_info : list[str]

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list[Any](set(celex_ids))    

    for i , celex_id in enumerate (celex_ids):
       full_doc_info = f"""Doc{i}:\n  {laws[laws['CELEX'] == celex_id]['act_raw_text'].iloc[0]} """ 

    return full_doc_info    



In [ ]:
search_docs("Drug dealing Sentences")

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("Drug dealing Sentences")

retrived = []

for i , doc in enumerate (docs):
    retrived.append([f"Doc{i}:"  doc.metadata['status','treaty','act_type','celex']])


[Document(metadata={'subject_matter': 'sources and branches of the law;  European Union law;  justice;  criminal law', 'status': 'In Force', 'legal_basis': '12002M031; 12002M034', 'additional_info': 'CNS 2001/0114', 'chunk_number': 1, 'eurovoc': 'penal code; Community law - national law; criminal procedure; penalty; drug traffic', 'treaty': 'TEU (1992)', 'act_type': 'Decision_FRAMW', 'cites': 'joint_action/1997/396; 31999Y0123%2801%29; joint_action/1998/733', 'celex': '32004F0757', 'authors': 'European Council', 'act_name': 'Council Framework Decision 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the field of illicit drug trafficking', 'document_length': 13663, 'total_chunks': 6}, page_content="11.11.2004 EN Official Journal of the European Union L 335/8 COUNCIL FRAMEWORK DECISION 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the 

In [ ]:
retrived = []

for doc in docs:
    retrived.append({doc.metadata['status','treaty']})

In [45]:
docs[0].metadata['celex']

'32004F0757'

In [ ]:
def get_celex_ids(docs):
    celex_ids = []
    for i in docs:
        celex_ids.append(docs[i].metadata['celex'])

    celex_ids = list(set(celex_ids))

    return celex_ids    

In [1]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI."),
    ("human", "Answer this question: {question}")
])

formatted = prompt.invoke({"question": "What is AI?"})
print(formatted)

messages=[SystemMessage(content='You are a helpful AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Answer this question: What is AI?', additional_kwargs={}, response_metadata={})]


In [9]:
laws[laws['CELEX'] == '31997R2046']['act_raw_text'].iloc[0]   

"Avis juridique important|31997R2046Council Regulation (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addiction Official Journal L 287 , 21/10/1997 P. 0001 - 0005COUNCIL REGULATION (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addictionTHE COUNCIL OF THE EUROPEAN UNION,Having regard to the Treaty establishing the European Community, and in particular Article 130w thereof,Having regard to the proposal from the Commission (1),Acting in accordance with the procedure laid down in Article 189c of the Treaty (2),Whereas the impact on the structures of a developing society of an economy based on the production of drugs, or which derives a substantial revenue from them, undermines a country's smooth integration into the world economy;Whereas the breakdown of social structures in developing countries due to drug consumption and the related industry is detrimental to sustainable social de